# Reproducible, Explainable ML Benchmark for Online Shoppers' Purchasing Intention

**Wagino, Muhammad Subhan, Silvia Ratna, M. Muflih** — Universitas Islam Kalimantan Muhammad Arsyad Al Banjari, Banjarmasin, Indonesia
*DECODE: Jurnal Pendidikan Teknologi Informasi* (to appear)

This single notebook reproduces every table and figure in the paper:
comparative modelling, XGBoost tuning, structural ablation, feature-selection
sensitivity, SHAP explainability, nested cross-validation, threshold and
probability calibration, bootstrap confidence intervals, and the **RAID**
(Resampling-Aware Inflation Decomposition) audit.

**How to run:** place the dataset CSV in `Dataset/` (see the repo `Dataset/README.md`),
then *Run All*. Tables are saved to `Results/` and figures to `Results/figures/`.
A single seed (`42`) controls every random step.

In [ ]:
# If running on a fresh environment (e.g. Google Colab), uncomment:
# !pip -q install numpy==2.1.3 pandas==2.2.3 scikit-learn==1.6.1 xgboost==3.4.1 \
#     catboost==1.2.10 imbalanced-learn==0.14.2 shap==0.52.0 matplotlib

In [ ]:
import os, warnings, platform
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from scipy import sparse
from scipy.stats import randint, uniform
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             brier_score_loss, confusion_matrix,
                             classification_report)
from sklearn.calibration import calibration_curve
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

## 0. Configuration
Everything controlling reproducibility lives in this cell.

In [ ]:
SEED = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
N_BOOTSTRAP = 2000

# --- paths (match the GitHub repo layout) ---
DATA_DIR = Path("Dataset")
RESULTS_DIR = Path("Results")
FIG_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "online_shoppers_intention.csv"  # auto-detects any CSV in Dataset/
TARGET = "Revenue"

CATEGORICAL_FEATURES = ["Month", "OperatingSystems", "Browser", "Region",
                        "TrafficType", "VisitorType", "Weekend"]
NUMERICAL_FEATURES = ["Administrative", "Administrative_Duration",
                      "Informational", "Informational_Duration",
                      "ProductRelated", "ProductRelated_Duration",
                      "BounceRates", "ExitRates", "PageValues", "SpecialDay"]
TEMPORAL_FEATURES = ["Month", "Weekend", "SpecialDay"]

K_FEATURES = 20
K_GRID = [10, 15, 20, 25, 30]
THRESHOLD_GRID = [round(0.10 + 0.05 * i, 2) for i in range(17)]  # 0.10..0.90

# Tuned XGBoost hyper-parameters reported in the paper (regenerate via Section 3).
TUNED_XGB_PARAMS = dict(n_estimators=617, max_depth=6, learning_rate=0.0231724683,
                        subsample=0.7359596102, colsample_bytree=0.7410275425,
                        gamma=0.1828687549, min_child_weight=1, reg_alpha=0, reg_lambda=1)
BASE_XGB_PARAMS = dict(n_estimators=300, max_depth=6, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8)
CATBOOST_PARAMS = dict(iterations=300, depth=6, learning_rate=0.05,
                       loss_function="Logloss", eval_metric="AUC",
                       auto_class_weights="Balanced", verbose=0)

# Set True to reuse the paper's tuned params (fast); False to re-run the search.
USE_PRESET_PARAMS = True

PRIMARY = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]
def save(df, name):
    p = RESULTS_DIR / name; df.to_csv(p, index=False); print("saved ->", p)

In [ ]:
# Software environment (reported for reproducibility)
import sklearn, xgboost, catboost, imblearn, shap
env = pd.DataFrame([
    ("Python", platform.python_version()), ("numpy", np.__version__),
    ("pandas", pd.__version__), ("scikit-learn", sklearn.__version__),
    ("xgboost", xgboost.__version__), ("catboost", catboost.__version__),
    ("imbalanced-learn", imblearn.__version__), ("shap", shap.__version__),
    ("matplotlib", matplotlib.__version__),
], columns=["Component", "Version"])
print(env.to_string(index=False)); save(env, "environment.csv")

## 1. Load data & leakage-safe preprocessing
The target is coerced to {0,1}; coded identifiers are declared categorical so they are not treated as ordinal. All preprocessing is fit inside pipelines on the training partition only.

In [ ]:
path = DATASET_PATH
if not path.exists():
    cands = sorted(DATA_DIR.glob("*.csv"))
    assert cands, f"No CSV found in {DATA_DIR}/ (see Dataset/README.md)"
    path = cands[0]
df = pd.read_csv(path); print("Loaded:", path, "| shape:", df.shape)

if df[TARGET].dtype == bool:
    df[TARGET] = df[TARGET].astype(int)
else:
    m = {"TRUE":1,"True":1,"true":1,1:1,"FALSE":0,"False":0,"false":0,0:0}
    df[TARGET] = df[TARGET].map(m).fillna(df[TARGET]).astype(int)
for c in CATEGORICAL_FEATURES:
    df[c] = df[c].astype(str)

X = df.drop(columns=[TARGET]); y = df[TARGET]
print("Target distribution:\n", y.value_counts(normalize=True).round(4) * 100)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
print("train:", X_train.shape, "| test:", X_test.shape)

In [ ]:
def build_preprocessor(numerical=NUMERICAL_FEATURES, categorical=CATEGORICAL_FEATURES, scale=False):
    """Median impute numerics (+optional scaling); most-frequent impute + one-hot categoricals."""
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scaler", StandardScaler()))
    cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", Pipeline(num_steps), numerical),
                              ("cat", cat, categorical)])

def spw(y_):
    neg, pos = np.bincount(np.asarray(y_).astype(int)); return neg / pos

def evaluate(name, y_true, y_pred, y_prob, key="Model"):
    return {key: name,
            "Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1-score": f1_score(y_true, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, y_prob),
            "PR-AUC": average_precision_score(y_true, y_prob)}

def metrics_at_threshold(y_true, y_prob, t=0.50):
    y_pred = (np.asarray(y_prob) >= t).astype(int)
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1-score": f1_score(y_true, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_true, y_prob),
            "PR-AUC": average_precision_score(y_true, y_prob),
            "Brier Score": brier_score_loss(y_true, y_prob)}

## 2. Comparative benchmark (Table 6)
Four classifiers under identical honest conditions. Logistic Regression is standardised; tree models are not; CatBoost handles categoricals natively.

In [ ]:
def make_base_xgb(spw_val=None):
    return XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                         random_state=SEED, scale_pos_weight=spw_val, **BASE_XGB_PARAMS)

rows = []
comp_models = {
    "Logistic Regression": Pipeline([("preprocessor", build_preprocessor(scale=True)),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))]),
    "Random Forest": Pipeline([("preprocessor", build_preprocessor(scale=False)),
        ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                         random_state=SEED, n_jobs=-1))]),
    "XGBoost": Pipeline([("preprocessor", build_preprocessor(scale=False)),
        ("model", make_base_xgb(spw(y_train)))]),
}
for name, pipe in comp_models.items():
    pipe.fit(X_train, y_train)
    rows.append(evaluate(name, y_test, pipe.predict(X_test), pipe.predict_proba(X_test)[:, 1]))

# CatBoost (native categoricals)
Xtr_cb, Xte_cb = X_train.copy(), X_test.copy()
for c in CATEGORICAL_FEATURES:
    Xtr_cb[c] = Xtr_cb[c].astype(str); Xte_cb[c] = Xte_cb[c].astype(str)
cb = CatBoostClassifier(random_seed=SEED, **CATBOOST_PARAMS)
cb.fit(Xtr_cb, y_train, cat_features=CATEGORICAL_FEATURES)
rows.append(evaluate("CatBoost", y_test, cb.predict(Xte_cb), cb.predict_proba(Xte_cb)[:, 1]))

comparative_df = pd.DataFrame(rows).sort_values("F1-score", ascending=False).reset_index(drop=True)
comparative_df[PRIMARY] = comparative_df[PRIMARY].round(4)
save(comparative_df, "table6_comparative.csv"); comparative_df

## 3. XGBoost hyper-parameter tuning
`USE_PRESET_PARAMS=True` reuses the paper's tuned values (fast). Set it to `False` to regenerate them with RandomizedSearchCV (30 iters, F1, 5-fold).

In [ ]:
if USE_PRESET_PARAMS:
    best_xgb_params = dict(TUNED_XGB_PARAMS)
    print("Using preset tuned params:", best_xgb_params)
else:
    tune_pipe = Pipeline([("preprocessor", build_preprocessor(scale=False)),
                          ("model", make_base_xgb(spw(y_train)))])
    param_dist = {"model__n_estimators": randint(300, 901),
                  "model__max_depth": randint(4, 9),
                  "model__learning_rate": uniform(0.01, 0.08),
                  "model__subsample": uniform(0.7, 0.3),
                  "model__colsample_bytree": uniform(0.6, 0.4),
                  "model__min_child_weight": randint(1, 6),
                  "model__gamma": uniform(0.0, 0.6),
                  "model__reg_lambda": [1, 2, 5, 10],
                  "model__reg_alpha": [0, 0.1, 0.5, 1.0]}
    search = RandomizedSearchCV(tune_pipe, param_dist, n_iter=30, scoring="f1",
        cv=StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED),
        n_jobs=-1, random_state=SEED, refit=True, verbose=1)
    search.fit(X_train, y_train)
    best_xgb_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
    print("Best CV F1:", round(search.best_score_, 4)); print(best_xgb_params)

def make_tuned_xgb(spw_val=None):
    return XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                         random_state=SEED, scale_pos_weight=spw_val, **best_xgb_params)

def final_pipeline(spw_val, k=K_FEATURES):
    return Pipeline([("preprocessor", build_preprocessor(scale=False)),
                     ("feature_selection", SelectKBest(mutual_info_classif, k=k)),
                     ("model", make_tuned_xgb(spw_val))])

## 4. Structural ablation (Table 7)
Tuned XGBoost with components added/removed: baseline, +feature selection (the final model), +SMOTE (training only), −temporal features, −PageValues.

In [ ]:
def fit_eval(pipe, Xtr, ytr, Xte, yte, name):
    pipe.fit(Xtr, ytr)
    return evaluate(name, yte, pipe.predict(Xte), pipe.predict_proba(Xte)[:, 1], key="Setting")

abl = []
abl.append(fit_eval(Pipeline([("preprocessor", build_preprocessor(scale=False)),
                              ("model", make_base_xgb(spw(y_train)))]),
                    X_train, y_train, X_test, y_test, "XGBoost Baseline"))
abl.append(fit_eval(final_pipeline(spw(y_train)), X_train, y_train, X_test, y_test,
                    "XGBoost + Feature Selection"))
abl.append(fit_eval(ImbPipeline([("preprocessor", build_preprocessor(scale=False)),
                                 ("smote", SMOTE(random_state=SEED)),
                                 ("model", make_base_xgb(None))]),
                    X_train, y_train, X_test, y_test, "XGBoost + SMOTE"))
temp = [c for c in TEMPORAL_FEATURES if c in X_train.columns]
abl.append(fit_eval(Pipeline([("preprocessor", build_preprocessor(
        [c for c in NUMERICAL_FEATURES if c not in temp],
        [c for c in CATEGORICAL_FEATURES if c not in temp], scale=False)),
        ("model", make_tuned_xgb(spw(y_train)))]),
    X_train.drop(columns=temp), y_train, X_test.drop(columns=temp), y_test,
    "XGBoost - Temporal Features"))
abl.append(fit_eval(Pipeline([("preprocessor", build_preprocessor(
        [c for c in NUMERICAL_FEATURES if c != "PageValues"], CATEGORICAL_FEATURES, scale=False)),
        ("model", make_tuned_xgb(spw(y_train)))]),
    X_train.drop(columns=["PageValues"]), y_train,
    X_test.drop(columns=["PageValues"]), y_test, "XGBoost - PageValues"))

ablation_df = pd.DataFrame(abl).sort_values("F1-score", ascending=False).reset_index(drop=True)
ablation_df[PRIMARY] = ablation_df[PRIMARY].round(4)
save(ablation_df, "table7_ablation.csv"); ablation_df

## 5. Feature-selection k-sensitivity (Table 8)
Is k=20 competitive across the grid?

In [ ]:
ks = []
for k in K_GRID:
    pipe = final_pipeline(spw(y_train), k=k); pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]
    r = evaluate(f"k={k}", y_test, pipe.predict(X_test), prob, key="k"); r["k"] = k
    ks.append(r)
k_df = pd.DataFrame(ks); k_df[PRIMARY] = k_df[PRIMARY].round(4); k_df = k_df[["k"] + PRIMARY]
save(k_df, "table8_k_sensitivity.csv")
plt.figure(figsize=(7, 4)); plt.plot(k_df["k"], k_df["F1-score"], marker="o")
plt.title("Feature-selection k-sensitivity"); plt.xlabel("k"); plt.ylabel("F1-score")
plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(FIG_DIR / "k_sensitivity.png", dpi=300, bbox_inches="tight"); plt.show()
k_df

## 6. SHAP explainability (Table 9 + figures)
Global importance (beeswarm + bar) and one local waterfall for the final tuned XGBoost + feature-selection model.

In [ ]:
import shap
final = final_pipeline(spw(y_train)); final.fit(X_train, y_train)
prep, sel, model = (final.named_steps["preprocessor"],
                    final.named_steps["feature_selection"], final.named_steps["model"])
names = prep.get_feature_names_out()[sel.get_support()]
Xte_sel = sel.transform(prep.transform(X_test))
if sparse.issparse(Xte_sel): Xte_sel = Xte_sel.toarray()
Xte_sel = pd.DataFrame(Xte_sel, columns=names)
sample = Xte_sel.sample(n=min(500, len(Xte_sel)), random_state=SEED)

explainer = shap.TreeExplainer(model); sv = explainer.shap_values(sample)
plt.figure(); shap.summary_plot(sv, sample, show=False); plt.tight_layout()
plt.savefig(FIG_DIR / "shap_summary.png", dpi=300, bbox_inches="tight"); plt.show()
plt.figure(); shap.summary_plot(sv, sample, plot_type="bar", show=False); plt.tight_layout()
plt.savefig(FIG_DIR / "shap_bar.png", dpi=300, bbox_inches="tight"); plt.show()

shap_imp = (pd.DataFrame({"Feature": sample.columns, "MeanAbsSHAP": np.abs(sv).mean(axis=0)})
            .sort_values("MeanAbsSHAP", ascending=False).reset_index(drop=True))
save(shap_imp, "table9_shap_importance.csv"); shap_imp.head(15)

In [ ]:
# Local SHAP waterfall for one confident positive prediction
prob_test = final.predict_proba(X_test)[:, 1]; pred_test = (prob_test >= 0.50).astype(int)
tp = np.where((y_test.to_numpy() == 1) & (pred_test == 1))[0]
pos = tp[0] if len(tp) else int(np.argmax(prob_test))
local = explainer(Xte_sel.iloc[[pos]])
shap.plots.waterfall(local[0], max_display=15, show=False)
plt.tight_layout(); plt.savefig(FIG_DIR / "shap_local_waterfall.png", dpi=300, bbox_inches="tight"); plt.show()

## 7. Nested cross-validation (Table 10)
Outer 5-fold evaluation with an inner 5-fold RandomizedSearch, to show the result is not split-specific.

In [ ]:
outer = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
inner = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
param_dist = {"model__n_estimators": randint(300, 901), "model__max_depth": randint(4, 9),
              "model__learning_rate": uniform(0.01, 0.08), "model__subsample": uniform(0.7, 0.3),
              "model__colsample_bytree": uniform(0.6, 0.4), "model__min_child_weight": randint(1, 6),
              "model__gamma": uniform(0.0, 0.6), "model__reg_lambda": [1, 2, 5, 10],
              "model__reg_alpha": [0, 0.1, 0.5, 1.0]}
nested = []
for fold, (tr, te) in enumerate(outer.split(X, y), start=1):
    Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
    pipe = Pipeline([("preprocessor", build_preprocessor(scale=False)),
                     ("feature_selection", SelectKBest(mutual_info_classif, k=K_FEATURES)),
                     ("model", make_base_xgb(spw(ytr)))])
    s = RandomizedSearchCV(pipe, param_dist, n_iter=30, scoring="f1", cv=inner,
                           n_jobs=-1, random_state=SEED, refit=True)
    s.fit(Xtr, ytr)
    r = evaluate(f"Fold {fold}", yte, s.predict(Xte), s.predict_proba(Xte)[:, 1], key="Fold")
    nested.append(r); print(f"Fold {fold} done")
nested_df = pd.DataFrame(nested); nested_df[PRIMARY] = nested_df[PRIMARY].round(4)
summary = pd.DataFrame({"Metric": PRIMARY,
                        "Mean": [nested_df[m].mean().round(4) for m in PRIMARY],
                        "Std": [nested_df[m].std().round(4) for m in PRIMARY]})
save(nested_df, "table10_nested_cv_folds.csv"); save(summary, "table10_nested_cv_summary.csv")
summary

## 8. Validation-based threshold tuning (Table 11)
The operating threshold is chosen on a validation split carved from the training set, then applied to the untouched test set.

In [ ]:
X_tr_sub, X_val, y_tr_sub, y_val = train_test_split(
    X_train, y_train, test_size=TEST_SIZE, stratify=y_train, random_state=SEED)
vp = final_pipeline(spw(y_tr_sub)); vp.fit(X_tr_sub, y_tr_sub)
val_prob = vp.predict_proba(X_val)[:, 1]
best_t, best_f1 = 0.50, -1
for t in THRESHOLD_GRID:
    f = f1_score(y_val, (val_prob >= t).astype(int), zero_division=0)
    if f > best_f1: best_f1, best_t = f, t
print("Selected threshold:", best_t)

final.fit(X_train, y_train)                     # refit on full training set
y_prob_test = final.predict_proba(X_test)[:, 1] # reused in Sections 9 & 10
thr_rows = []
for label, t in [("Default (0.50)", 0.50), (f"Validation-selected ({best_t:.2f})", best_t)]:
    m = {k: round(v, 4) for k, v in metrics_at_threshold(y_test, y_prob_test, t).items()}
    thr_rows.append({"Setting": label, **m})
thr_df = pd.DataFrame(thr_rows); save(thr_df, "table11_threshold.csv"); thr_df

## 9. Probability calibration (Brier score + reliability)

In [ ]:
brier = brier_score_loss(y_test, y_prob_test); print("Brier score:", round(brier, 6))
prob_true, prob_pred = calibration_curve(y_test, y_prob_test, n_bins=10, strategy="uniform")
rel = pd.DataFrame({"Mean Predicted": prob_pred.round(4), "Observed Frequency": prob_true.round(4)})
save(rel, "calibration_reliability.csv")
save(pd.DataFrame([{"Model": "Tuned XGBoost + FS", "Brier Score": round(brier, 6)}]), "calibration_brier.csv")
plt.figure(figsize=(6, 6)); plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
plt.plot(prob_pred, prob_true, marker="o", label="Model")
plt.title("Reliability curve"); plt.xlabel("Mean predicted probability")
plt.ylabel("Observed frequency"); plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "calibration_curve.png", dpi=300, bbox_inches="tight"); plt.show()

## 10. Bootstrap 95% confidence intervals (Table 12)

In [ ]:
def bootstrap_ci(y_true, y_prob, t=0.50, n=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed); yt = np.asarray(y_true); yp = np.asarray(y_prob); N = len(yt)
    point = metrics_at_threshold(yt, yp, t); samp = {m: [] for m in point}
    for _ in range(n):
        idx = rng.integers(0, N, N); a, b = yt[idx], yp[idx]
        if a.sum() == 0 or a.sum() == len(a): continue
        for m, v in metrics_at_threshold(a, b, t).items(): samp[m].append(v)
    return pd.DataFrame([{"Metric": m, "Point Estimate": round(point[m], 4),
                          "95% CI Lower": round(float(np.percentile(samp[m], 2.5)), 4),
                          "95% CI Upper": round(float(np.percentile(samp[m], 97.5)), 4)} for m in point])

ci_def = bootstrap_ci(y_test, y_prob_test, 0.50); ci_def.insert(0, "Setting", "Default (0.50)")
ci_sel = bootstrap_ci(y_test, y_prob_test, best_t); ci_sel.insert(0, "Setting", f"Validation-selected ({best_t:.2f})")
boot_df = pd.concat([ci_def, ci_sel], ignore_index=True); save(boot_df, "table12_bootstrap_ci.csv")

pm = ci_def[ci_def["Metric"].isin(PRIMARY)].copy()
plt.figure(figsize=(8, 5))
plt.errorbar(pm["Metric"], pm["Point Estimate"],
             yerr=[pm["Point Estimate"] - pm["95% CI Lower"], pm["95% CI Upper"] - pm["Point Estimate"]],
             fmt="o", capsize=5)
plt.title("Bootstrap 95% CIs (primary metrics)"); plt.xticks(rotation=20); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "bootstrap_ci.png", dpi=300, bbox_inches="tight"); plt.show()
boot_df

## 11. RAID — Resampling-Aware Inflation Decomposition (Tables 13-14)

The key contribution. Three arms with the **same** model and preprocessing:

| Arm | Description |
|-----|-------------|
| **A**  | Leaky: SMOTE on ALL data, then split; scored on the balanced, contaminated test fold |
| **A'** | The arm-A model, scored on the natural (real) hold-out test set |
| **B**  | Honest: SMOTE inside the training pipeline only; natural test |

Decomposition per metric: `M_A − M_B = (M_A − M_A') + (M_A' − M_B)` =
test-distribution artefact + synthetic contamination.

In [ ]:
# Arm B: honest
honest = ImbPipeline([("preprocessor", build_preprocessor(scale=False)),
                      ("smote", SMOTE(random_state=SEED)),
                      ("model", make_tuned_xgb(None))])
honest.fit(X_train, y_train)
res_b = evaluate("B: Honest (SMOTE in training, natural test)",
                 y_test, honest.predict(X_test), honest.predict_proba(X_test)[:, 1], key="Arm")

# Arms A & A': leaky preprocessing + SMOTE BEFORE the split (one shared model)
prep_leak = build_preprocessor(scale=False)
X_all = prep_leak.fit_transform(X)                       # fit on all data (leak)
X_res, y_res = SMOTE(random_state=SEED).fit_resample(X_all, y)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_res, y_res, test_size=TEST_SIZE, stratify=y_res, random_state=SEED)
mA = make_tuned_xgb(None); mA.fit(Xr_tr, yr_tr)
res_a = evaluate("A: Leaky (resample before split, balanced test)",
                 yr_te, mA.predict(Xr_te), mA.predict_proba(Xr_te)[:, 1], key="Arm")
Xte_leak = prep_leak.transform(X_test)                   # natural test via leaky prep
res_ap = evaluate("A': Leaky training, natural test",
                  y_test, mA.predict(Xte_leak), mA.predict_proba(Xte_leak)[:, 1], key="Arm")

raid_df = pd.DataFrame([res_a, res_ap, res_b]); raid_df[PRIMARY] = raid_df[PRIMARY].round(4)
save(raid_df, "table13_raid_arms.csv"); raid_df

In [ ]:
# Additive decomposition
dec = []
for m in ["F1-score", "ROC-AUC", "PR-AUC"]:
    total = res_a[m] - res_b[m]; art = res_a[m] - res_ap[m]; cont = res_ap[m] - res_b[m]
    pct = lambda p: f"{round(100*p/total)}%" if total else "n/a"
    dec.append({"Metric": m, "Total inflation (A-B)": round(total, 4),
                "Test-distribution artefact (A-A')": f"{round(art,4)} ({pct(art)})",
                "Synthetic contamination (A'-B)": f"{round(cont,4)} ({pct(cont)})"})
decomp_df = pd.DataFrame(dec); save(decomp_df, "table14_raid_decomposition.csv")

plt.figure(figsize=(6, 4))
plt.bar(["A: Leaky", "B: Honest"], [res_a["F1-score"], res_b["F1-score"]],
        color=["#d62728", "#2ca02c"])
for i, v in enumerate([res_a["F1-score"], res_b["F1-score"]]):
    plt.text(i, v + 0.01, f"{v:.4f}", ha="center")
plt.ylim(0, 1); plt.ylabel("F1-score (positive class)")
plt.title("Leakage inflates F1: SMOTE-before-split vs honest split")
plt.tight_layout(); plt.savefig(FIG_DIR / "leakage_ab.png", dpi=300, bbox_inches="tight"); plt.show()
decomp_df

## Done

All tables are in `Results/` and all figures in `Results/figures/`.

**Citation**
```bibtex
@article{wagino2026raid,
  title   = {A Reproducible, Explainable Machine Learning Benchmark for
             Online Shoppers' Purchasing Intention},
  author  = {Wagino and Subhan, Muhammad and Ratna, Silvia and Muflih, M.},
  journal = {DECODE: Jurnal Pendidikan Teknologi Informasi},
  year    = {2026}, note = {To appear}
}
```
Released under the MIT License.